# 03 — Feature Engineering

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Construir um DataFrame agregado por usuário com features que descrevem o comportamento financeiro. Este arquivo será o input do modelo de ML.

---

## 0. Setup

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/transactions_clean.csv')

# convertendo a data para garantir que funcione
df['Date'] = pd.to_datetime(df['Date'])

print('Shape:', df.shape)
print("Agora sim! Dados carregados ✅")
df.head(3)

Shape: (19046, 11)
Agora sim! Dados carregados ✅


,User_ID,Date,Description,Amount,Transaction Type,Category,Account Name,Year,Month,Month_Name,Period
0,USER_0001,2018-01-01,Amazon,11.11,debit,Shopping,credit card,2018,1,Jan,2018-01
1,USER_0001,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,checking,2018,1,Jan,2018-01
2,USER_0001,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,credit card,2018,1,Jan,2018-01


## 1. Agregações Mensais por Usuário

### Por que excluir essas categorias?

Nem todo débito é **gasto real**. Algumas categorias são movimentações financeiras internas que
simplesmente transferem dinheiro entre contas do próprio usuário, incluí-las distorceria a análise:

| Categoria excluída | Motivo |
|---|---|
| `Credit Card Payment` | Pagamento da fatura — o gasto real já foi contabilizado quando a compra ocorreu |
| `Savings Transfer` | Transferência para poupança — não é despesa, é alocação de patrimônio |
| `Transfer` | Transferência entre contas — movimentação interna sem impacto no consumo |
| `Investment` | Aporte em investimentos — idem: alocação de patrimônio, não gasto |

> **Impacto:** Excluindo essas 4 categorias, `Rent` emerge claramente como o maior gasto real (35% do total),
> e os cálculos de `savings_rate` e `Saldo_Mensal_Real` passam a refletir a saúde financeira real do usuário.


In [11]:
# filtrando débitos e removendo movimentação entre contas ou pagamento de fatura 
categorias_excluir = ['Credit Card Payment', 'Savings Transfer', 'Transfer', 'Investment']

df_gastos_reais = df[
    (df['Transaction Type'] == 'debit') & 
    (~df['Category'].isin(categorias_excluir))
].copy()


In [12]:

# criando o resumo mensal básico por usuário

features_basicas = df_gastos_reais.groupby(['User_ID', 'Period']).agg(
    Total_Gasto=('Amount', 'sum'),
    Media_Transacao=('Amount', 'mean'),
    Qtd_Transacoes=('Amount', 'count')
).reset_index()

# ajustando nomes para ficar organizado
features_basicas.columns = ['User_ID', 'Period', 'Total_Gasto', 'Media_Transacao', 'Qtd_Transacoes']

# criando a visão de Crédito vs Débito 
pivot_cred_deb = df.pivot_table(index=['User_ID', 'Period'], columns='Transaction Type', 
                                values='Amount', aggfunc='sum', fill_value=0).reset_index()

# mesclando as duas tabelas
features_completas = pd.merge(features_basicas, pivot_cred_deb, on=['User_ID', 'Period'])

# criando a coluna de saldo final
features_completas['Saldo_Mensal'] = features_completas.get('credit', 0) - features_completas.get('debit', 0)

print(f"Shape: {features_completas.shape}")

print("Estrutura de features criada 📊 ")
features_completas.head()

Shape: (1221, 8)
Estrutura de features criada 📊 


,User_ID,Period,Total_Gasto,Media_Transacao,Qtd_Transacoes,credit,debit,Saldo_Mensal
0,USER_0001,2018-01,2066.65,82.666000,25,7162.89,2931.45,4231.44
1,USER_0001,2018-02,2089.44,87.060000,24,5220.75,3165.05,2055.70
2,USER_0001,2018-03,2178.66,60.518333,36,7321.50,3500.16,3821.34
3,USER_0001,2018-04,2862.66,75.333158,38,7166.88,6029.54,1137.34
4,USER_0001,2018-05,10507.56,318.410909,33,5091.55,11392.03,-6300.48


In [13]:
# saldo mensal real (receita - despesa)
# total recebido (crédito) por usuário + período
# Credit Card Payment é pagamento de fatura, não receita real, entao será removido
    
receita_mensal = df[(df['Transaction Type'] == 'credit') & 
                    (df['Description'] != 'Credit Card Payment')].groupby(['User_ID', 'Period'])['Amount'].sum().reset_index()
receita_mensal.columns = ['User_ID', 'Period', 'Total_Receita']

# juntando com features_completas
features_completas = features_completas.merge(receita_mensal, on=['User_ID', 'Period'], how='left').fillna({'Total_Receita': 0})

# calculando o saldo
features_completas['Saldo_Mensal_Real'] = (features_completas['Total_Receita'] - features_completas['Total_Gasto']).round(2)

print("Saldo mensal real criado!✅")
features_completas[['User_ID', 'Period', 'Total_Receita', 'Total_Gasto', 'Saldo_Mensal_Real']].head()

Saldo mensal real criado!✅


,User_ID,Period,Total_Receita,Total_Gasto,Saldo_Mensal_Real
0,USER_0001,2018-01,4000.0,2066.65,1933.35
1,USER_0001,2018-02,4000.0,2089.44,1910.56
2,USER_0001,2018-03,6000.0,2178.66,3821.34
3,USER_0001,2018-04,4000.0,2862.66,1137.34
4,USER_0001,2018-05,4000.0,10507.56,-6507.56


## 2. Features por Usuário

In [14]:
# --- Features de saldo e receita/despesa ---


monthly = features_completas[['User_ID', 'Period', 'debit', 'credit', 'Total_Receita', 'Saldo_Mensal_Real']].copy()
monthly['Saldo_Calculado'] = monthly['Total_Receita'] - monthly['debit']

feat_saldo = monthly.groupby('User_ID').agg(
    avg_monthly_debit   = ('debit',        'mean'), # média de quanto sai
    avg_monthly_credit = ('Total_Receita', 'mean'), # média de quanto entra
    avg_saldo           = ('Saldo_Mensal_Real', 'mean'), # lucro médio do usuário
    std_saldo           = ('Saldo_Mensal_Real', 'std'),  # desvio padrão
    spending_volatility = ('debit',        'std'),  # grau de flutuação
    pct_meses_negativo  = ('Saldo_Mensal_Real', lambda x: round((x < 0).mean(), 3)), # qtd de meses com saldo negativo
    total_meses         = ('Saldo_Mensal_Real', 'count')
).reset_index()

# taxa de poupança média = (receita - despesa) / receita
feat_saldo['savings_rate'] = ( # quanto o usuário consegue guardar
    (feat_saldo['avg_monthly_credit'] - feat_saldo['avg_monthly_debit']) /
    feat_saldo['avg_monthly_credit'].replace(0, np.nan)
).round(3)

print(f'feat_saldo OK: {feat_saldo.shape[0]} usuarios x {feat_saldo.shape[1]-1} features')
feat_saldo.head()


feat_saldo OK: 51 usuarios x 8 features


,User_ID,avg_monthly_debit,avg_monthly_credit,avg_saldo,std_saldo,spending_volatility,pct_meses_negativo,total_meses,savings_rate
0,USER_0001,4575.418095,4464.285714,1462.265714,2878.127111,2576.685869,0.095,21,-0.025
1,USER_0002,3092.826667,4136.631250,1474.228333,2588.383099,636.790311,0.167,24,0.252
2,USER_0003,3367.328333,3124.895000,213.397083,1729.746365,687.632079,0.250,24,-0.078
3,USER_0004,2943.169583,3349.859583,406.690000,1338.018613,694.305041,0.208,24,0.121
4,USER_0005,2737.333333,4842.969167,3144.224167,1783.159641,661.454993,0.083,24,0.435


**O que as features de saldo revelam:**

| Feature | Mín | Média | Máx |
|---|---|---|---|
| `savings_rate` | -2,30 | +0,14 | +0,65 |
| `pct_meses_negativo` | 0,0% | 31,4% | 100,0% |
| `spending_volatility` | US$ 289 | US$ 754 | US$ 1.633 |

- **savings_rate negativo** significa gastar mais do que se recebe, cerca de 14 usuários estão nessa situação
- **pct_meses_negativo = 1,0** (100%) aparece em 5 usuários, onde todos os meses estão no vermelho, sem exceção
- A **volatilidade de gastos** varia 5,6x entre o usuário mais previsível e o mais errático

In [15]:
# --- Feature da variação de despesas mês a mês ---

features_completas = features_completas.sort_values(['User_ID', 'Period'])

features_completas['Variacao_Gasto_Mensal'] = (
    features_completas
    .groupby('User_ID')['Total_Gasto']  # agrupa por usuário
    .pct_change()                        # calcula a variação % em relação à linha anterior
    * 100                                # transforma em porcentagem
).round(2)

print("Variação mensal criada!✅")
features_completas[['User_ID', 'Period', 'Total_Gasto', 'Variacao_Gasto_Mensal']].head(10)
# Nota: o primeiro mês de cada usuário sempre sera NaN, tendo em vista que não há mês anterior pata comparação

Variação mensal criada!✅


,User_ID,Period,Total_Gasto,Variacao_Gasto_Mensal
0,USER_0001,2018-01,2066.65,NaN
1,USER_0001,2018-02,2089.44,1.10
2,USER_0001,2018-03,2178.66,4.27
3,USER_0001,2018-04,2862.66,31.40
4,USER_0001,2018-05,10507.56,267.06
5,USER_0001,2018-06,2115.05,-79.87
6,USER_0001,2018-07,2302.64,8.87
7,USER_0001,2018-08,1992.68,-13.46
8,USER_0001,2018-09,2314.74,16.16
9,USER_0001,2018-10,2242.37,-3.13


In [16]:
# --- Features de principais categorias (apenas débitos) ---

debits = df[df['Transaction Type'] == 'debit']

total_by_user = debits.groupby('User_ID')['Amount'].sum().rename('total_debit')

# % do gasto na principal categoria, onde o usuário mais gasta
top_cat = (debits.groupby(['User_ID', 'Category'])['Amount'].sum()
           .reset_index()
           .sort_values('Amount', ascending=False)
           .groupby('User_ID')
           .first()
           .reset_index()
           .rename(columns={'Category': 'top_category', 'Amount': 'top_category_spend'}))

top_cat = top_cat.merge(total_by_user, on='User_ID')
top_cat['top_category_pct'] = (top_cat['top_category_spend'] / top_cat['total_debit']).round(3)

# definição de categorias fixas (recorrentes)
FIXED_CATS = ['Rent', 'Phone Bill', 'Internet Bill', 'Insurance', 'Utilities']
fixed = (debits[debits['Category'].isin(FIXED_CATS)]
         .groupby('User_ID')['Category']
         .nunique()
         .rename('num_fixed_expenses')
         .reset_index())

# tem investimento?
has_inv = (debits[debits['Category'] == 'Investment']
           .groupby('User_ID')['Amount']
           .sum()
           .rename('total_investment')
           .reset_index())
has_inv['has_investment'] = 1

print('Features de categoria OK')

perfil_usuario = top_cat.merge(fixed, on='User_ID', how='left').merge(has_inv, on='User_ID', how='left').fillna(0)

perfil_usuario.head(10)



Features de categoria OK


,User_ID,top_category,top_category_spend,total_debit,top_category_pct,num_fixed_expenses,total_investment,has_investment
0,USER_0001,Credit Card Payment,33041.36,96083.78,0.344,1,0.00,0.0
1,USER_0002,Rent,31977.54,74227.84,0.431,5,0.00,0.0
2,USER_0003,Rent,31397.66,80815.88,0.389,5,0.00,0.0
3,USER_0004,Rent,31144.12,70636.07,0.441,5,0.00,0.0
4,USER_0005,Rent,20816.93,65696.00,0.317,5,5516.94,1.0
5,USER_0006,Rent,22584.41,54499.36,0.414,5,4252.04,1.0
6,USER_0007,Rent,44946.75,164566.27,0.273,5,0.00,0.0
7,USER_0008,Rent,32457.88,75769.25,0.428,5,0.00,0.0
8,USER_0009,Rent,21817.59,73998.86,0.295,5,5996.80,1.0
9,USER_0010,Rent,31579.04,76657.14,0.412,5,0.00,0.0


**Comportamento de alocação:**
- **15 usuários (29%) realizam investimentos**, esses serão o cluster "Investidor Estratégico" no NB04
- **36 usuários (71%) não investem**, a ausência de investimento é um dos sinais mais fortes do modelo
- Categoria dominante mais comum: `Rent`, aparece como top_category para a maioria dos usuários,
  confirmando que habitação é o maior compromisso financeiro individual

In [17]:
# --- Features de todas as categorias (apenas débitos) ---

# somando gasto por usuário + período + categoria
gasto_por_categoria = df_gastos_reais.groupby(
    ['User_ID', 'Period', 'Category']
)['Amount'].sum().reset_index()

# pivot - cada categoria vira uma coluna
gasto_categorias_wide = gasto_por_categoria.pivot_table(
    index=['User_ID', 'Period'],
    columns='Category',
    values='Amount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# renomear colunas (padrão: gasto_nome_categoria)
gasto_categorias_wide.columns.name = None
novas_colunas = []
for col in gasto_categorias_wide.columns:
    if col in ['User_ID', 'Period']:
        novas_colunas.append(col)
    else:
        novo_nome = 'Gasto_' + col.replace(' ', '_').replace('&', '').replace('__', '_')
        novas_colunas.append(novo_nome)
gasto_categorias_wide.columns = novas_colunas

# juntando com as features básicas
features_completas = features_completas.merge(
    gasto_categorias_wide,
    on=['User_ID', 'Period'],
    how='left'
)

print(f"Features de todas as categoria criadas!✅") 
print(f"Shape final: {features_completas.shape}")

features_completas.head()



Features de todas as categoria criadas!✅
Shape final: (1221, 53)


,User_ID,Period,Total_Gasto,Media_Transacao,Qtd_Transacoes,credit,debit,Saldo_Mensal,Total_Receita,Saldo_Mensal_Real,...,Gasto_Rent,Gasto_Restaurants,Gasto_Rideshare,Gasto_Shopping,Gasto_Software_Apps,Gasto_Streaming_Services,Gasto_Taxes,Gasto_Television,Gasto_Travel,Gasto_Utilities
0,USER_0001,2018-01,2066.65,82.666000,25,7162.89,2931.45,4231.44,4000.0,1933.35,...,0.0,156.80,0.0,100.37,0.0,0.0,0.0,0.0,0.0,140.0
1,USER_0001,2018-02,2089.44,87.060000,24,5220.75,3165.05,2055.70,4000.0,1910.56,...,0.0,290.85,0.0,11.11,0.0,0.0,0.0,0.0,0.0,160.0
2,USER_0001,2018-03,2178.66,60.518333,36,7321.50,3500.16,3821.34,6000.0,3821.34,...,0.0,234.05,0.0,73.85,0.0,0.0,0.0,0.0,0.0,147.0
3,USER_0001,2018-04,2862.66,75.333158,38,7166.88,6029.54,1137.34,4000.0,1137.34,...,0.0,202.27,0.0,54.47,0.0,0.0,0.0,0.0,0.0,125.0
4,USER_0001,2018-05,10507.56,318.410909,33,5091.55,11392.03,-6300.48,4000.0,-6507.56,...,0.0,127.30,0.0,219.45,0.0,0.0,0.0,0.0,0.0,125.0


In [18]:
# --- Uso de cartão de crédito ---

gasto_credito = df[
    (df['Transaction Type'] == 'debit') &   # só gastos
    (df['Account Name'].str.lower() == 'credit card')   # só os feitos no crédito
].groupby(['User_ID', 'Period'])['Amount'].sum().reset_index()

gasto_credito.columns = ['User_ID', 'Period', 'Total_Gasto_Credito']

# merge com features_completas (que já tem o total_gasto)
features_completas = features_completas.merge(
    gasto_credito,
    on=['User_ID', 'Period'],
    how='left'
)

features_completas['Total_Gasto_Credito'] = features_completas['Total_Gasto_Credito'].fillna(0)  # se não usou crédito naquele mês, é 0

# calculando a porcentagem: (quanto foi no crédito / total gasto) * 100
features_completas['Perc_Gasto_Credito'] = (
    (features_completas['Total_Gasto_Credito'] / features_completas['Total_Gasto']) * 100
).round(2)

print("Métrica de dependência de crédito criada! ✅")
features_completas[['User_ID', 'Period', 'Total_Gasto_Credito', 'Perc_Gasto_Credito']].head()

Métrica de dependência de crédito criada! ✅


,User_ID,Period,Total_Gasto_Credito,Perc_Gasto_Credito
0,USER_0001,2018-01,519.76,25.15
1,USER_0001,2018-02,517.49,24.77
2,USER_0001,2018-03,619.71,28.44
3,USER_0001,2018-04,1250.71,43.69
4,USER_0001,2018-05,873.95,8.32


**Uso do cartão de crédito:**
- Média geral: **9,5% do gasto total** feito no cartão
- Nenhum usuário ultrapassa 60%, não há perfil de "dependência de crédito" no dataset
- Usuários Em Risco têm uso de cartão ligeiramente maior (10,4%) que Investidores (5,8%)

> O cartão não é o vilão, o problema está no volume absoluto de gastos e na ausência de poupança.

In [19]:
# --- features mensais para nível de usuário ---
# features_completas está no nível mensal (1221 linhas), vamos resumir para 1 linha por usuário (51 linhas)

agg_monthly = features_completas.groupby('User_ID').agg(
    avg_variacao_gasto_mensal = ('Variacao_Gasto_Mensal', 'mean'),
    std_variacao_gasto_mensal = ('Variacao_Gasto_Mensal', 'std'),
    avg_perc_gasto_credito    = ('Perc_Gasto_Credito',    'mean'),
    avg_qtd_transacoes_mes    = ('Qtd_Transacoes',        'mean') 
).reset_index()

# Arredondamento e limpeza
cols_resumo = ['avg_variacao_gasto_mensal', 'std_variacao_gasto_mensal', 
               'avg_perc_gasto_credito', 'avg_qtd_transacoes_mes']

agg_monthly[cols_resumo] = agg_monthly[cols_resumo].round(3).fillna(0)

print(f'agg_monthly OK: {agg_monthly.shape[0]} usuários. ✅')
agg_monthly.head()

agg_monthly OK: 51 usuários. ✅


,User_ID,avg_variacao_gasto_mensal,std_variacao_gasto_mensal,avg_perc_gasto_credito,avg_qtd_transacoes_mes
0,USER_0001,26.680,107.927,26.847,29.381
1,USER_0002,0.916,35.722,18.579,13.000
2,USER_0003,5.676,33.242,15.882,13.000
3,USER_0004,7.862,47.723,18.416,14.000
4,USER_0005,18.217,82.600,17.318,12.000


In [20]:
# --- Dominância de categoria por mês ---

idx_dominante = df[df['Transaction Type'] == 'debit'].groupby(['User_ID', 'Period'])['Amount'].idxmax()
categoria_dominante = df.loc[idx_dominante, ['User_ID', 'Period', 'Category', 'Amount']]

categoria_dominante.columns = ['User_ID', 'Period', 'Categoria_Dominante', 'Valor_Dominante']

features_completas = features_completas.merge(
    categoria_dominante,
    on=['User_ID', 'Period'],
    how='left'
)

# criando métrica de "peso da dominante" (% do gasto total que essa categoria levou)
features_completas['Perc_Dominante'] = (
    (features_completas['Valor_Dominante'] / features_completas['Total_Gasto']) * 100
).round(2)

print("Feature de Categoria Dominante Mensal criada! 🏆")
features_completas[['User_ID', 'Period', 'Categoria_Dominante', 'Perc_Dominante']].head(10)

Feature de Categoria Dominante Mensal criada! 🏆


,User_ID,Period,Categoria_Dominante,Perc_Dominante
0,USER_0001,2018-01,Mortgage & Rent,60.36
1,USER_0001,2018-02,Mortgage & Rent,59.70
2,USER_0001,2018-03,Mortgage & Rent,57.26
3,USER_0001,2018-04,Mortgage & Rent,43.58
4,USER_0001,2018-05,Home Improvement,76.14
5,USER_0001,2018-06,Mortgage & Rent,58.98
6,USER_0001,2018-07,Mortgage & Rent,54.17
7,USER_0001,2018-08,Mortgage & Rent,62.60
8,USER_0001,2018-09,Mortgage & Rent,53.89
9,USER_0001,2018-10,Mortgage & Rent,53.92


## 3. Consolidar todas as features

In [21]:
# --- todas as features em 1 linha por usuário ---

user_features = feat_saldo.copy()

for nome, df_feat in [
    ('perfil_usuario', perfil_usuario),
    ('agg_monthly',    agg_monthly),
]:
    antes = user_features.shape[1]
    user_features = user_features.merge(df_feat, on='User_ID', how='left')
    print(f'  [{nome}] +{user_features.shape[1] - antes} colunas -> total: {user_features.shape[1]-1}')

print(f'\nShape final: {user_features.shape[0]} usuarios x {user_features.shape[1]-1} features')

  [perfil_usuario] +7 colunas -> total: 15
  [agg_monthly] +4 colunas -> total: 19

Shape final: 51 usuarios x 19 features


In [22]:
# --- última checagem de qualidade ---

nulls = user_features.isnull().sum()
print('Nulos por coluna:')
if nulls.sum() > 0:
    print(nulls[nulls > 0])
else:
    print('  Nenhum nulo encontrado. ✅')
print()
print('Colunas geradas:')
for col in user_features.columns[1:]:
    print(f'  {col}')

Nulos por coluna:
  Nenhum nulo encontrado. ✅

Colunas geradas:
  avg_monthly_debit
  avg_monthly_credit
  avg_saldo
  std_saldo
  spending_volatility
  pct_meses_negativo
  total_meses
  savings_rate
  top_category
  top_category_spend
  total_debit
  top_category_pct
  num_fixed_expenses
  total_investment
  has_investment
  avg_variacao_gasto_mensal
  std_variacao_gasto_mensal
  avg_perc_gasto_credito
  avg_qtd_transacoes_mes


In [23]:
# --- preview ordenado por savings_rate ---

# mostrando o espectro do mais poupador ao mais endividado

PREVIEW_COLS = [
    'User_ID', 'savings_rate', 'pct_meses_negativo',
    'avg_saldo', 'spending_volatility',
    'top_category', 'top_category_pct',
    'has_investment', 'num_fixed_expenses',
    'avg_perc_gasto_credito', 'avg_variacao_gasto_mensal'
]

user_features[PREVIEW_COLS].sort_values('savings_rate', ascending=False).reset_index(drop=True)

,User_ID,savings_rate,pct_meses_negativo,avg_saldo,spending_volatility,top_category,top_category_pct,has_investment,num_fixed_expenses,avg_perc_gasto_credito,avg_variacao_gasto_mensal
0,USER_0049,0.633,0.083,4912.587917,391.663060,Rent,0.351,1.0,5,25.910,5.401
1,USER_0014,0.632,0.042,4613.662083,351.586007,Rent,0.380,1.0,5,23.185,3.999
2,USER_0015,0.609,0.083,4166.825833,557.971474,Rent,0.381,1.0,5,22.134,11.313
3,USER_0044,0.606,0.083,4155.751250,451.417868,Rent,0.350,1.0,5,16.727,5.860
4,USER_0050,0.577,0.083,3731.225833,537.920610,Rent,0.372,1.0,5,26.098,9.410
5,USER_0039,0.563,0.042,3748.753750,484.953506,Rent,0.342,1.0,5,16.693,3.522
6,USER_0009,0.563,0.042,5034.245000,446.482043,Rent,0.295,1.0,5,20.210,-1.760
7,USER_0021,0.525,0.000,3164.029167,472.566364,Rent,0.381,1.0,5,18.763,11.004
8,USER_0042,0.525,0.208,3210.492083,357.368284,Rent,0.348,1.0,5,24.235,2.291
9,USER_0045,0.463,0.083,2590.490000,574.512663,Rent,0.345,1.0,5,30.218,13.226


**Espectro financeiro dos 51 usuários:**

O preview ordenado por `savings_rate` revela 3 grupos visualmente distintos, mesmo antes de qualquer algoritmo:

- **Topo da lista (savings_rate > 0,35):** todos têm `has_investment = 1`, `pct_meses_negativo` abaixo de 15%
- **Meio da lista (savings_rate entre 0,05 e 0,35):** poupam, mas não investem; meses negativos esporádicos
- **Base da lista (savings_rate < 0):** `pct_meses_negativo` acima de 60%, `has_investment = 0`

> Esses 3 grupos naturais são exatamente o que o K-Means irá formalizar no notebook 04.

## 4. Salvar

In [24]:
import os
os.makedirs('../data/processed', exist_ok=True)

user_features.to_csv('../data/processed/user_features.csv', index=False)
print('Features salvas em data/processed/user_features.csv')
print(f'{len(user_features)} usuarios x {user_features.shape[1]-1} features')
user_features.dtypes

print("Perfis definidos e ficheiro atualizado! ✅")

Features salvas em data/processed/user_features.csv
51 usuarios x 19 features
Perfis definidos e ficheiro atualizado! ✅
